In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re

# UBAH sesuai folder Anda
DRIVE_DIR = Path("/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD")

PREFIX = "train1"   # akan mencari file seperti ILtrain1_part<number>...

print("Folder:", DRIVE_DIR)
print("Ada file berapa:", len(list(DRIVE_DIR.iterdir())))

# Tampilkan file yang match pola ILtrain1_part<number> (bebas ekstensi)
pattern = re.compile(rf"^{re.escape(PREFIX)}_part(\d+)(\.gz)?$")

matches = []
for p in DRIVE_DIR.iterdir():
    if p.is_file():
        m = pattern.match(p.name)
        if m:
            matches.append((int(m.group(1)), p.name))

matches.sort(key=lambda x: x[0])

print("\nPart terdeteksi:")
for n, name in matches:
    print(n, name)

if not matches:
    print("\n⚠️ Tidak ada file yang match. Coba kirim 10 nama file di folder ini biar saya sesuaikan polanya.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder: /content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD
Ada file berapa: 6

Part terdeteksi:
1 train1_part1.gz
2 train1_part2.gz
3 train1_part3.gz
4 train1_part4.gz
5 train1_part5.gz


In [ ]:
from pathlib import Path
import re
import shutil

def combine_parts_binary_to_gdrive(input_dir, prefix, output_gz_path, delete_parts=False):
    """
    Combine file part {prefix}_part<number>(.gz optional) secara biner (rb/wb)
    dan simpan hasilnya ke Google Drive pada output_gz_path.
    """
    input_dir = Path(input_dir)
    output_gz_path = Path(output_gz_path)

    # Pastikan folder output ada
    output_gz_path.parent.mkdir(parents=True, exist_ok=True)

    # Ambil part yang match: ILtrain1_part1, ILtrain1_part2, ... (boleh ada .gz)
    pattern = re.compile(rf"^{re.escape(prefix)}_part(\d+)(\.gz)?$")

    parts = []
    for p in input_dir.iterdir():
        if p.is_file():
            m = pattern.match(p.name)
            if m:
                parts.append((int(m.group(1)), p))

    if not parts:
        raise FileNotFoundError(
            f"Tidak menemukan part files dengan pola {prefix}_part<number>(.gz) di {input_dir}"
        )

    # Sort by nomor part agar urut benar
    parts.sort(key=lambda x: x[0])
    part_numbers = [n for n, _ in parts]
    print("✅ Part terdeteksi & urut:", part_numbers)

    # Overwrite bila sudah ada file output
    if output_gz_path.exists():
        output_gz_path.unlink()

    print(f"🔧 Menggabungkan {len(parts)} file -> {output_gz_path}")
    with open(output_gz_path, "wb") as fout:
        for idx, (n, part_path) in enumerate(parts, start=1):
            print(f"  - [{idx}/{len(parts)}] {part_path.name}")
            with open(part_path, "rb") as fin:
                shutil.copyfileobj(fin, fout)

    print(f"✅ Combine selesai. Output tersimpan di Drive: {output_gz_path}")
    print(f"   Size: {output_gz_path.stat().st_size/1e6:.2f} MB")

    # Opsional hapus part
    if delete_parts:
        for _, part_path in parts:
            part_path.unlink()
        print("🗑️ Part files dihapus.")

    return output_gz_path

In [ ]:
# Folder part di Google Drive (ubah sesuai folder Anda)
DRIVE_PARTS_DIR = "/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD"

# Simpan hasil gabungan ke Google Drive juga
OUT_GZ_IN_DRIVE = "/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz"

combine_parts_binary_to_gdrive(
    input_dir=DRIVE_PARTS_DIR,
    prefix="train1",
    output_gz_path=OUT_GZ_IN_DRIVE,
    delete_parts=False
)

✅ Part terdeteksi & urut: [1, 2, 3, 4, 5]
🔧 Menggabungkan 5 file -> /content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz
  - [1/5] train1_part1.gz
  - [2/5] train1_part2.gz
  - [3/5] train1_part3.gz
  - [4/5] train1_part4.gz
  - [5/5] train1_part5.gz
✅ Combine selesai. Output tersimpan di Drive: /content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz
   Size: 74108.59 MB


PosixPath('/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz')

In [ ]:
import gzip

out_path = "/content/drive/MyDrive/dataset skripsi/dataset SIM4MXRD/ILtrain1.db.gz"

try:
    with gzip.open(out_path, "rb") as f:
        _ = f.read(1024)
    print("✅ File hasil gabungan valid gzip dan bisa dibuka.")
except Exception as e:
    print("❌ File hasil gabungan tidak valid gzip / belum urut benar:", repr(e))

✅ File hasil gabungan valid gzip dan bisa dibuka.
